<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'> Modificado en 2019-1 al 2024-2 por Equipo Docente IIC2233. </font>
</p>

# Tabla de contenidos

1. [Serialización de objetos](#Serialización-de-objetos)
    1. [`pickle`](#pickle)
        1. [Personalizar la serialización en `pickle`](#Personalizar-la-deserialización-en-pickle)
        2. [Personalizar la deserialización en `pickle`](#Personalizar-la-deserialización-en-pickle)
    2. [JSON](#JSON)
        1. [Personalizar la serialización en JSON](#Personalizar-la-serialización-en-JSON)
        2. [Personalizar la deserialización en JSON](#Personalizar-la-deserialización-en-JSON)

# Serialización de objetos

Toda la información que almacena un computador se guarda en base a *bits* (ceros y unos) y *bytes* (secuencias de 8 *bits*). Esta semana estudiaremos en detalle el uso y manejo de *bytes* en Python.

Imaginemos que buscamos guardar una estructura de datos o una instancia de una clase para volver a leerla más adelante o para comunicarla a otro programa. De alguna forma, estos datos deben ser guardados como una serie o secuencia de *bytes*. Aquí es cuando aparece el concepto de **serialización**.

Este concepto se refiere al procedimiento de transformar un objeto en una secuencia o serie de *bytes*. Esto nos permite almacenar la información y el estado de un objeto de forma persistente, por ejemplo en un archivo o una base de datos para poder consultarlo más tarde. También nos permite enviar el objeto a otros computadores y programas.

## `pickle`

El módulo `pickle` de Python permite guardar y cargar casi cualquier objeto de Python. Este módulo ofrece los siguientes métodos:

- `dumps`: serializa un objeto, es decir, lo **guarda** como una secuencia de bytes.
- `loads`: deserializa un objeto serializado, es decir, **carga** un objeto a su estado original.

Una vez que un objeto es serializado, este es persistente y está listo para volver a ser usado en el futuro por el mismo u otro programa.

In [1]:
import pickle


tupla = ("a", 1, 3, "hola")
tupla_serializada = pickle.dumps(tupla)

print(f"Resultado serialización: {tupla_serializada}")
print(f"Tipo de versión serializada: {type(tupla_serializada)}")

tupla_deserializada = pickle.loads(tupla_serializada)
print(f"Resultado deserialización: {tupla_deserializada}")

print()
print(f"Tupla original: {tupla}")
print(f"Tupla deserializada: {tupla_deserializada}")
print(f"¿Las tuplas son iguales? {tupla == tupla_deserializada}")
print(f"¿Las tuplas son el mismo objeto? {tupla is tupla_deserializada}")


Resultado serialización: b'\x80\x04\x95\x13\x00\x00\x00\x00\x00\x00\x00(\x8c\x01a\x94K\x01K\x03\x8c\x04hola\x94t\x94.'
Tipo de versión serializada: <class 'bytes'>
Resultado deserialización: ('a', 1, 3, 'hola')

Tupla original: ('a', 1, 3, 'hola')
Tupla deserializada: ('a', 1, 3, 'hola')
¿Las tuplas son iguales? True
¿Las tuplas son el mismo objeto? False


Además, `pickle` nos ofrece los métodos `dump` y `load` (casi el mismo nombre que los anteriores, pero sin la *s*). Estos métodos también serializan y deserializan, pero **a través de archivos**: 

- `dump`: guarda un archivo con el objeto serializado.
- `load`: deserializa un objeto almacenado en un archivo.

In [2]:
from os import path

lista = [1, 2, 3, 7, 8, 3]

with open(path.join("data", "mi_lista.bin"), "wb") as file:
    pickle.dump(lista, file)

with open(path.join("data", "mi_lista.bin"), "rb") as file:
    lista_cargada = pickle.load(file)

print(f"Lista original: {lista}")
print(f"Lista cargada : {lista_cargada}")
print(f"¿Las listas son iguales? {lista == lista_cargada}")
print(f"¿Las listas son el mismo objeto? {lista is lista_cargada}")


Lista original: [1, 2, 3, 7, 8, 3]
Lista cargada : [1, 2, 3, 7, 8, 3]
¿Las listas son iguales? True
¿Las listas son el mismo objeto? False


### Importante
`pickle` es un módulo no seguro. Esto significa que **nunca** debes cargar un archivo *pickle* cuando no conoces su procedencia, ya que éste podría ejecutar código malicioso en tu computador. No entraremos en detalles sobre cómo inyectar código a través del módulo `pickle`, pero si te interesa puedes revisar este [enlace](https://checkoway.net/musings/pickle/) donde se demuestra su uso malicioso.

### Personalizar la serialización en `pickle`

Cuando `pickle` trata de serializar un objeto, lo primero que hará es verificar que el objeto que se quiere serializar sea de una clase que tenga implementado el método `__getstate__`. Este método debe retornar un diccionario con los atributos que se quieren serializar. Si `__getstate__` no estuviese implementado, entonces `pickle` guardará el atributo `__dict__` del objeto. 

El atributo `__dict__` es un diccionario que guarda todos los atributos y métodos de un objeto. En otras palabras, `objeto.atributo` es equivalente a `objeto.__dict__["atributo"]` y `objeto.atributo = 42` es equivalente a `objeto.__dict__["atributo"] = 42`.

El implementar el método `__getstate__` nos permite personalizar la serialización del objeto. Usando este método podemos crear un diccionario que contenga solo la información que deseamos guardar. 

In [3]:
class Persona:

    def __init__(self, nombre, edad):
        self.nombre = nombre
        self.edad = edad
        self.mensaje = "No pasa nada"

    def __getstate__(self):
        """
        Retorna el estado actual del objeto, para que sea serializado por pickle

        Aquí creamos una copia del diccionario actual, para modificar la copia 
        y no el objeto original
        """
        # Usamos una copia del diccionario original para alterar solo la copia
        nueva = self.__dict__.copy()
        # Modificamos un atributo en el objeto serializado.
        # Sin embargo, el objeto original no ha cambiado.
        nueva.update({"mensaje": "¡Me están serializando!"})
        # Lo que retornemos es lo que será serializado por pickle
        return nueva


In [4]:
original = Persona("Juan", 30)
print(f"Mensaje original: {original.mensaje}")
serializado = pickle.dumps(original)
deserializado = pickle.loads(serializado)

# El objeto original sigue igual
print(f"Mensaje original: {original.mensaje}")
print(f"Mensaje deserializado: {deserializado.mensaje}")


Mensaje original: No pasa nada
Mensaje original: No pasa nada
Mensaje deserializado: ¡Me están serializando!


### Personalizar la deserialización en `pickle`

De forma análoga, podemos personalizar la **deserialización**. Para esto debemos implementar el método `__setstate__`, que se ejecutará cada vez que llamemos a `load` o `loads`. El método `__setstate__` recibe como argumento el diccionario que representa el estado del objeto que fue serializado. Luego debe asignarlo al diccionario del objeto `self.__dict__ = diccionario_con_estado`. Esto no impide que se realicen otras acciones que modifiquen `diccionario_con_estado` antes o después de la asignación.

Si el método `__setstate__` no estuviese implementado, entonces se asignará al `__dict__` del objeto el estado deserializado sin realizar otras acciones adicionales.

In [5]:
class Persona:

    def __init__(self, nombre, edad):
        self.nombre = nombre
        self.edad = edad
        self.mensaje = "No pasa nada"

    def __getstate__(self):
        nueva = self.__dict__.copy()
        print(f"[__getstate__] Serializando a {nueva['nombre']}")
        nueva.update({"mensaje": "¡Me están serializando!"})
        # Lo que retornemos es lo que será serializado por pickle
        return nueva

    def __setstate__(self, state):
        print("[__setstate__] Objeto recién deserializado, actualizando su estado")
        # Al desarializar modificamos el estado
        state.update({"nombre": f"{state['nombre']} deserializado"})
        self.__dict__ = state


In [6]:
original = Persona("Juan", 30)
print(f"Nombre original: {original.nombre}")
# Al usar pickle.dumps() se ejecuta el método __getstate__
serializado = pickle.dumps(original)


Nombre original: Juan
[__getstate__] Serializando a Juan


In [7]:
print("¡Ejecutar loads → deserializar!")
# Al usar pickle.loads() se ejecuta el método __setstate__
deserializado = pickle.loads(serializado)
print(f"Nombre deserializado: {deserializado.nombre}, y su mensaje: {deserializado.mensaje}")


¡Ejecutar loads → deserializar!
[__setstate__] Objeto recién deserializado, actualizando su estado
Nombre deserializado: Juan deserializado, y su mensaje: ¡Me están serializando!


Una aplicación de los métodos `__getstate__` y `__setstate__` es cuando necesitamos serializar un objeto que contiene un atributo que depende de las condiciones actuales del programa.

Por ejemplo, imaginemos que un objeto que guarda información sobre los usuarios conectados actualmente al programa, como la cantidad de usuarios y la información correspondiente a la conexión con cada uno. Cuando guardamos el objeto, deberíamos eliminar estas conexiones, ya que al cargarlo en otra instancia del programa no deberíamos poder comunicarnos con los usuarios de la instancia anterior. Para lograr esto usamos el método `__getstate__`. 

Similarmente, cuando se cargue el mismo objeto desde el archivo serializado, será necesario volver a crear las conexiones con las condiciones del programa nuevo. Para realizar esto tendremos que implementar `__setstate__`.

## JSON

Una de las desventajas de los objetos serializados con `pickle` es que sólo pueden ser deserializados por otros programas escritos en Python. En cambio, **JSON** (JavaScript Object Notation) es un formato de texto estándar de intercambio de datos que puede ser interpretado por muchos lenguajes, y por ende es algo ampliamente utilizado para traspasar información de un programa a otro (por ejemplo, la comunicación entre dos computadores mediante internet). JSON además es *human-readable*, es decir, puede ser fácilmente leído y entendido por humanos.

**El formato en que almacena la información es similar, pero no igual, a los diccionarios de Python.**

En JSON solo es posible serializar instancias de `int`, `str`, `float`, `dict`, `bool`, `list`, `tuple` y `NoneType`, de acuerdo a esta tabla de transformación que puedes revisar en [este link](https://docs.python.org/3.7/library/json.html#encoders-and-decoders). Por defecto no es posible serializar funciones o instancias de otras clases.

En Python, existe un módulo llamado `json` que provee métodos para serializar objetos en el  formato JSON. Este módulo provee una interfaz similar a la de `pickle`, es decir, los métodos `dump`(`s`) y `load`(`s`).

In [8]:
from itertools import count
import json


class Persona:
    id = count()

    def __init__(self, nombre, edad, estado_civil):
        self.nombre = nombre
        self.edad = edad
        self.estado_civil = estado_civil
        self.id_ = next(self.id)


In [9]:
p = Persona("Juan", 35, "Soltero")

json_string = json.dumps(p.__dict__)
print("Datos en formato JSON:", type(json_string), json_string)

json_deserializado = json.loads(json_string)
print("Datos en formato Python:", type(json_deserializado), json_deserializado)


Datos en formato JSON: <class 'str'> {"nombre": "Juan", "edad": 35, "estado_civil": "Soltero", "id_": 0}
Datos en formato Python: <class 'dict'> {'nombre': 'Juan', 'edad': 35, 'estado_civil': 'Soltero', 'id_': 0}


### Personalizar la serialización en JSON

Cuando queremos guardar un objeto como JSON podemos personalizar la transformación utilizando un `json.JSONEncoder`, de forma análoga a como lo hicimos con `__getstate__`. Para esto debemos crear una clase que hereda de la clase `json.JSONEncoder` y sobreescribir el método `default`:

In [10]:
from datetime import datetime


class PersonaEncoder(json.JSONEncoder):

    def default(self, obj):
        """Serializa en forma personalizada el objeto de tipo Persona"""
        if isinstance(obj, Persona):
            return {
                "Persona_id": obj.id_,
                "nombre": obj.nombre,
                "edad": obj.edad,
                "estado_civil": obj.estado_civil,
                "ano_nacimiento": datetime.now().year - obj.edad,
            }
        # Mantenemos la serialización por defecto para otros tipos
        return super().default(obj)


# Creamos tres instancias
p1 = Persona("Juanita", 37, "Soltera")
p2 = Persona("Jorge", 33, "Casado")
p3 = Persona("Mariela", 24, "Soltera")


Probamos la serialización usando el método por defecto:

In [11]:
json_string = json.dumps(p1.__dict__)

print(json_string)


{"nombre": "Juanita", "edad": 37, "estado_civil": "Soltera", "id_": 1}


Ahora, comparemos el resultado al serializar usando nuestro método personalizado:

In [12]:
json_string = json.dumps(p1, cls=PersonaEncoder)
print(json_string)

json_string = json.dumps(p2, cls=PersonaEncoder)
print(json_string)

json_string = json.dumps(p3, cls=PersonaEncoder)
print(json_string)


{"Persona_id": 1, "nombre": "Juanita", "edad": 37, "estado_civil": "Soltera", "ano_nacimiento": 1987}
{"Persona_id": 2, "nombre": "Jorge", "edad": 33, "estado_civil": "Casado", "ano_nacimiento": 1991}
{"Persona_id": 3, "nombre": "Mariela", "edad": 24, "estado_civil": "Soltera", "ano_nacimiento": 2000}


### Personalizar la deserialización en JSON

Cuando queremos transformar un JSON a un objeto en Python podemos utilizar los `object_hook`, de forma análoga a como lo hicimos con `__setstate__`. 

El `object_hook` es un parámetro de los métodos `load` y `loads`, que espera una función que sea capaz de manejar un diccionario y retorne un objeto de Python. En el proceso de deserialización, todo objeto JSON será convertido a un diccionario de Python, y luego pasado a la función `object_hook` para hacer la transformación final.

Por ejemplo, si queremos cargar datos JSON en una lista de tuplas en vez de un diccionario:

In [13]:
def funcion_hook(diccionario):
    return [(key, value) for key, value in diccionario.items()]


json_string = '{"nombre": "Jorge", "edad": 34, "estado_civil": "casado", "puntaje": 90.5}'
datos = json.loads(json_string, object_hook=funcion_hook)

print(datos)


[('nombre', 'Jorge'), ('edad', 34), ('estado_civil', 'casado'), ('puntaje', 90.5)]


Es importante recordar que **todo objeto de JSON es convertido a un diccionario**, y luego entregado al `object_hook`. Esto tiene implicancias cuando tenemos un JSON con objetos anidados:

In [14]:
def funcion_hook(diccionario):
    """Esta función transforma un diccionario en un número"""
    print(f"Me llegó el diccionario: {diccionario}")
    valor = 33 if len(diccionario) > 1 else 42
    print(f"Lo transformaré en {valor}\n")
    return valor


json_string = '{"a": {"b": 1, "c": [2, 3, {}], "d": null, "e": {"f": true}}}'
datos = json.loads(json_string, object_hook=funcion_hook)

print(datos)


Me llegó el diccionario: {}
Lo transformaré en 42

Me llegó el diccionario: {'f': True}
Lo transformaré en 42

Me llegó el diccionario: {'b': 1, 'c': [2, 3, 42], 'd': None, 'e': 42}
Lo transformaré en 33

Me llegó el diccionario: {'a': 33}
Lo transformaré en 42

42


¿Qué pasó ahí arriba? Lo primero que podemos notar es que el proceso de conversión funciona de adentro hacia afuera, y además que el resultado que entregue `object_hook` se "arrastra" en el proceso.